# Notebook 05 — MOID Ablation Experiment

**Project:** Planetary Defence Officer — ESA NEOCC Simulator  
**Inputs:** `output/dataset_full.npz`, `output/feature_names.json`, `output/scaler.pkl`, `output/rf_full.pkl`  
**Outputs:** `output/dataset_noMOID.npz`, `output/rf_noMOID.pkl`, `output/feature_importances.json`, `output/feature_importance_full.png`, `output/feature_importance_noMOID.png`, `output/metrics.json` (extended)

**Purpose:** Quantify how much the model relies on MOID (Minimum Orbit Intersection Distance)
as a single feature. This mirrors the SDSS redshift experiment in STELLaRUM: one leviathan
feature that the classifier leans on so heavily that its removal causes significant performance
collapse. How much, exactly, is what this notebook measures.

**Method:** Drop the `Minimum Orbit Intersection` column, re-fit a new StandardScaler
on the reduced training set (never touching test), retrain with the winning Phase 3
configuration (Borderline-SMOTE → RF), evaluate with the same metric set.

Random state: `42`.

---

## 1. Load artifacts from prior phases

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import json, pickle, os

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score, roc_auc_score,
)
from imblearn.over_sampling import BorderlineSMOTE

os.makedirs('output', exist_ok=True)

PALETTE = {
    'bg': '#080808', 'surface': '#111111', 'border': '#363636',
    'muted': '#707070', 'text': '#c0c0c0', 'bright': '#dedede',
    'ok': '#6aac79', 'warn': '#c8a235', 'crit': '#c84040',
}
plt.rcParams.update({
    'figure.facecolor': PALETTE['bg'], 'axes.facecolor': PALETTE['bg'],
    'axes.edgecolor': PALETTE['border'], 'text.color': PALETTE['text'],
    'axes.labelcolor': PALETTE['text'], 'xtick.color': PALETTE['muted'],
    'ytick.color': PALETTE['muted'], 'grid.color': '#1a1a1a',
    'grid.alpha': 0.6, 'font.family': 'monospace',
    'axes.titlecolor': PALETTE['bright'], 'axes.titlesize': 11,
    'legend.facecolor': '#080808', 'legend.edgecolor': '#363636',
    'legend.labelcolor': '#c0c0c0',
})

data = np.load('output/dataset_full.npz', allow_pickle=True)
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']

with open('output/feature_names.json') as f:
    feature_names = json.load(f)

with open('output/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('output/rf_full.pkl', 'rb') as f:
    rf_full = pickle.load(f)

with open('output/metrics.json') as f:
    metrics_out = json.load(f)

moid_idx = feature_names.index('Minimum Orbit Intersection')
print(f"MOID column index : {moid_idx}")
print(f"Feature names     : {feature_names}")
print(f"\nFull dataset — train: {X_train.shape}  test: {X_test.shape}")

## 2. Build no-MOID dataset

We inverse-transform the already-scaled arrays to recover original units, drop the MOID
column, then fit a **new** StandardScaler on the reduced training set only.
This is correct: if we simply dropped the MOID column from the scaled arrays, the remaining
columns' scales would be unchanged — but a fresh scaler fit ensures proper normalisation
and avoids any implicit dependency on MOID's scale during model training.

In [ ]:
# Inverse transform → drop MOID column → re-scale
X_train_raw = scaler.inverse_transform(X_train)
X_test_raw  = scaler.inverse_transform(X_test)

X_train_noMOID_raw = np.delete(X_train_raw, moid_idx, axis=1)
X_test_noMOID_raw  = np.delete(X_test_raw,  moid_idx, axis=1)

feature_names_noMOID = [f for f in feature_names if f != 'Minimum Orbit Intersection']
print(f"Features without MOID ({len(feature_names_noMOID)}):")
for i, f in enumerate(feature_names_noMOID, 1):
    print(f"  {i:2d}. {f}")

# Re-fit scaler on training data only
scaler_noMOID = StandardScaler()
X_train_noMOID = scaler_noMOID.fit_transform(X_train_noMOID_raw)
X_test_noMOID  = scaler_noMOID.transform(X_test_noMOID_raw)

np.savez(
    'output/dataset_noMOID.npz',
    X_train=X_train_noMOID, X_test=X_test_noMOID,
    y_train=y_train, y_test=y_test,
    feature_names=np.array(feature_names_noMOID),
)
print("\n✓ output/dataset_noMOID.npz saved")

## 3. Train RF on no-MOID data (winning Phase 3 config)

Same configuration as the winner in notebook 04: Borderline-SMOTE → RF(n=200).
The only difference is 12 features instead of 13.

In [ ]:
print("Applying Borderline-SMOTE to no-MOID training set...")
smote = BorderlineSMOTE(random_state=42, kind='borderline-1')
X_sm_noMOID, y_sm_noMOID = smote.fit_resample(X_train_noMOID, y_train)
print(f"After SMOTE: {X_sm_noMOID.shape}  hazardous: {y_sm_noMOID.sum()}")

rf_noMOID = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_noMOID.fit(X_sm_noMOID, y_sm_noMOID)

with open('output/rf_noMOID.pkl', 'wb') as f:
    pickle.dump(rf_noMOID, f)
print("✓ output/rf_noMOID.pkl saved")

## 4. Evaluate both models — side-by-side

In [ ]:
def get_metrics(model, X_te, y_te, label, slug):
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    acc     = accuracy_score(y_te, y_pred)
    pr_auc  = average_precision_score(y_te, y_prob)
    roc_auc = roc_auc_score(y_te, y_prob)
    cm      = confusion_matrix(y_te, y_pred)
    rep     = classification_report(y_te, y_pred,
                                    target_names=['SAFE','HAZARDOUS'],
                                    output_dict=True)

    fig, ax = plt.subplots(figsize=(4.5, 3.8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['SAFE','HAZARDOUS'],
                yticklabels=['SAFE','HAZARDOUS'],
                linewidths=1, linecolor=PALETTE['border'],
                annot_kws={'color': PALETTE['bright'], 'size': 14})
    ax.set_title(f'◈ {label}', pad=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout()
    plt.savefig(f'output/confusion_ablation_{slug}.png', dpi=150, bbox_inches='tight')
    plt.show()

    return {
        'accuracy':         round(acc, 4),
        'pr_auc_hazardous': round(pr_auc, 4),
        'roc_auc':          round(roc_auc, 4),
        'hazardous': {
            'precision': round(rep['HAZARDOUS']['precision'], 4),
            'recall':    round(rep['HAZARDOUS']['recall'], 4),
            'f1':        round(rep['HAZARDOUS']['f1-score'], 4),
        },
        'safe': {
            'precision': round(rep['SAFE']['precision'], 4),
            'recall':    round(rep['SAFE']['recall'], 4),
            'f1':        round(rep['SAFE']['f1-score'], 4),
        },
        'confusion_matrix': cm.tolist(),
    }, y_prob

m_full,   prob_full   = get_metrics(rf_full,   X_test,        y_test, 'RF WITH MOID (13 features)',    slug='full')
m_noMOID, prob_noMOID = get_metrics(rf_noMOID, X_test_noMOID, y_test, 'RF WITHOUT MOID (12 features)', slug='noMOID')

In [ ]:
# Comparison table
rows = [
    {'Model': 'RF full (with MOID)',    **{k: m_full[k]   for k in ['accuracy','pr_auc_hazardous','roc_auc']},
     'Haz Prec': m_full['hazardous']['precision'],   'Haz Recall': m_full['hazardous']['recall'],   'Haz F1': m_full['hazardous']['f1']},
    {'Model': 'RF no-MOID (12 feats)', **{k: m_noMOID[k] for k in ['accuracy','pr_auc_hazardous','roc_auc']},
     'Haz Prec': m_noMOID['hazardous']['precision'], 'Haz Recall': m_noMOID['hazardous']['recall'], 'Haz F1': m_noMOID['hazardous']['f1']},
]
df_cmp = pd.DataFrame(rows).set_index('Model')
print("◈ MOID ABLATION — PERFORMANCE COMPARISON")
print("=" * 80)
print(df_cmp.to_string())
print("=" * 80)

delta_acc    = m_noMOID['accuracy']         - m_full['accuracy']
delta_prauc  = m_noMOID['pr_auc_hazardous'] - m_full['pr_auc_hazardous']
delta_recall = m_noMOID['hazardous']['recall'] - m_full['hazardous']['recall']
delta_f1     = m_noMOID['hazardous']['f1']     - m_full['hazardous']['f1']

print(f"\nDelta (no-MOID minus full):")
print(f"  Accuracy     : {delta_acc:+.4f}")
print(f"  PR AUC (haz) : {delta_prauc:+.4f}   <- primary metric")
print(f"  Haz Recall   : {delta_recall:+.4f}")
print(f"  Haz F1       : {delta_f1:+.4f}")

## 5. Feature importances — both models

In [ ]:
def plot_importances(model, names, title, filename, highlight=None):
    importances = model.feature_importances_
    idx = np.argsort(importances)[::-1]
    sorted_names = [names[i] for i in idx]
    sorted_imp   = importances[idx]

    short = {
        'Minimum Orbit Intersection': 'MOID',
        'Absolute Magnitude':         'Abs. Mag. H',
        'Est Dia in KM(min)':         'Diam. min (km)',
        'Est Dia in KM(max)':         'Diam. max (km)',
        'Relative Velocity km per sec':'Rel. Vel.',
        'Miss Dist.(Astronomical)':   'Miss Dist. AU',
        'Orbit Uncertainity':         'Orbit Uncert.',
        'Jupiter Tisserand Invariant':'Tisserand TJ',
        'Eccentricity':               'Eccentricity',
        'Semi Major Axis':            'Semi-Major a',
        'Inclination':                'Inclination',
        'Perihelion Distance':        'Perihelion q',
        'Aphelion Dist':              'Aphelion Q',
    }
    disp_names = [short.get(n, n) for n in sorted_names]

    colors = []
    for n in sorted_names:
        if highlight and n == highlight:
            colors.append(PALETTE['crit'])
        elif sorted_imp[sorted_names.index(n)] > 0.15:
            colors.append(PALETTE['warn'])
        else:
            colors.append(PALETTE['ok'])

    fig, ax = plt.subplots(figsize=(8, max(4, len(names) * 0.42)))
    bars = ax.barh(range(len(sorted_names)), sorted_imp[::-1],
                   color=colors[::-1], edgecolor=PALETTE['border'], height=0.65)
    ax.set_yticks(range(len(sorted_names)))
    ax.set_yticklabels(disp_names[::-1], fontsize=9)
    ax.set_xlabel('Mean Decrease in Impurity (Gini importance)')
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.3)
    for spine in ax.spines.values():
        spine.set_edgecolor(PALETTE['border'])
    # Annotate each bar
    for i, (bar, val) in enumerate(zip(bars, sorted_imp[::-1])):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8, color=PALETTE['muted'])
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()
    return dict(zip(sorted_names, sorted_imp.tolist()))

print("Full model (with MOID):")
imp_full = plot_importances(
    rf_full, feature_names,
    '◈ FEATURE IMPORTANCES — RF with MOID (13 features)',
    'output/feature_importance_full.png',
    highlight='Minimum Orbit Intersection',
)
print()
print("No-MOID model:")
imp_noMOID = plot_importances(
    rf_noMOID, feature_names_noMOID,
    '◈ FEATURE IMPORTANCES — RF without MOID (12 features)',
    'output/feature_importance_noMOID.png',
)

In [ ]:
moid_importance = imp_full.get('Minimum Orbit Intersection', 0)
top_without_moid = sorted(imp_noMOID.items(), key=lambda x: -x[1])[:3]

print(f"MOID importance in full model : {moid_importance:.4f}  ({100*moid_importance:.1f}% of total)")
print(f"\nTop 3 features without MOID:")
for name, imp in top_without_moid:
    print(f"  {name:<35} {imp:.4f}  ({100*imp:.1f}%)")

with open('output/feature_importances.json', 'w') as f:
    json.dump({'full_model': imp_full, 'noMOID_model': imp_noMOID}, f, indent=2)
print("\n✓ output/feature_importances.json saved")

## 6. PR curves — with vs without MOID

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for label, prob, color, ls in [
    ('RF with MOID',    prob_full,   PALETTE['ok'],   '-'),
    ('RF without MOID', prob_noMOID, PALETTE['crit'], '--'),
]:
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    ax.plot(rec, prec, color=color, linestyle=ls, lw=2.5,
            label=f'{label}  AP={ap:.4f}')

baseline_rate = y_test.mean()
ax.axhline(baseline_rate, color=PALETTE['muted'], linestyle=':', lw=1,
           label=f'No-skill ({baseline_rate:.3f})')

ax.set_xlabel('Recall (hazardous)')
ax.set_ylabel('Precision (hazardous)')
ax.set_title('◈ PR CURVES — MOID ABLATION')
ax.legend(fontsize=10)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
for spine in ax.spines.values():
    spine.set_edgecolor(PALETTE['border'])
plt.tight_layout()
plt.savefig('output/pr_curves_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved output/pr_curves_ablation.png")

## 7. Physical interpretation

**Why is MOID so informative?**

MOID — the Minimum Orbit Intersection Distance — measures the closest distance between
the orbital paths of an asteroid and Earth, regardless of where either body is at any
given moment. It is not the distance at a particular close approach event; it is the
theoretical floor on how close an encounter between the two orbits *could* be.

The NASA Potentially Hazardous Asteroid (PHA) classification criterion is exactly:

```
MOID < 0.05 AU   AND   H < 22
```

Where 0.05 AU ≈ 7.5 million km — roughly 20 times the Earth–Moon distance. An object
with MOID below this threshold has an orbital geometry that *can* produce a dangerous
close approach given the right timing. H < 22 corresponds to a diameter of roughly
140m or larger, the threshold above which an impactor would cause regional destruction.

Because the `Hazardous` label in this dataset is the NASA PHA definition, and MOID is
one of the two defining variables, the classifier is essentially re-deriving a known
physical criterion from the data. This is not data leakage — MOID is a legitimate,
measurable orbital property. But it does explain why removing it causes performance
to collapse more sharply than removing any other single feature.

**What the ablation reveals:**

The performance delta (full model vs no-MOID, printed in cell 4) quantifies
the information MOID carries. The feature importance chart shows what rises to
replace it: typically absolute magnitude H, estimated diameter, and perihelion
distance (q = a(1-e)) — all physically motivated proxies for the two-part PHA
criterion. But no single substitute is as clean or as information-dense as MOID.

**Implication for game design:**

Real PHAs in this dataset have MOID values that are unambiguously below 0.05 AU.
The RF classifier reads MOID and is essentially certain. This is why the game
*cannot* rely on real PHAs for difficulty — they are too easy to classify.
The Borderline-SMOTE objects generated in notebook 06 will have MOID values near
the 0.05 AU boundary, where the classifier's confidence drops to 0.45–0.75 and
genuine uncertainty exists. Those are the objects that create gameplay tension.

## 8. Save metrics.json — moid_ablation section

In [ ]:
metrics_out['moid_ablation'] = {
    'full_model':   m_full,
    'noMOID_model': m_noMOID,
    'delta': {
        'accuracy':         round(delta_acc, 4),
        'pr_auc_hazardous': round(delta_prauc, 4),
        'haz_recall':       round(delta_recall, 4),
        'haz_f1':           round(delta_f1, 4),
    },
    'moid_importance_in_full_model': round(moid_importance, 4),
}

with open('output/metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("✓ output/metrics.json — moid_ablation section added")
print(f"\nPhase 4 complete.")
print(f"  MOID importance : {moid_importance:.4f} ({100*moid_importance:.1f}% of full model)")
print(f"  PR AUC drop     : {delta_prauc:+.4f}")
print(f"  Recall drop     : {delta_recall:+.4f}")